# quiz 1 基礎 + 進階

In [1]:
import torch
import importlib
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from other_code import quiz1

importlib.reload(quiz1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/root/venvs/torch-cu121/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


模型載入

In [3]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda:0")

Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.94s/it]


模型測試

In [5]:
model.eval()

messages = [
    {"role": "system", "content": "請使用繁體中文回答。"},
    {"role": "user", "content": "請用三句話介紹什麼是大型語言模型。"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)

大型語言模型是一種人工智慧技術，它能夠學習和理解大量的文本數據，從而生成合乎語法、邏輯且具有表達力的文本。這些模型通過深度學習方法訓練而成，能夠處理各種語言任務，如文本生成、摘要、對話系統等。


利用20道題目評估LLM

In [5]:
base_dir = Path.cwd()
print("目前工作資料夾：", base_dir)

with open(base_dir / "question/quiz1_baseline.json", "r", encoding="utf-8") as f: questions = json.load(f)
print(f"成功讀取 {len(questions)} 題")

目前工作資料夾： /root/MMIP/week7
成功讀取 20 題


In [ ]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_baseline.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "FP16",
    "prompt_setting": "baseline",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_baseline_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：10432.70 MiB
正在回答第2題：知識理解
本題平均顯存：10440.34 MiB
正在回答第3題：知識理解
本題平均顯存：10439.52 MiB
正在回答第4題：知識理解
本題平均顯存：10433.84 MiB
正在回答第5題：數學與邏輯
本題平均顯存：10433.80 MiB
正在回答第6題：數學與邏輯
本題平均顯存：10434.16 MiB
正在回答第7題：數學與邏輯
本題平均顯存：10434.43 MiB
正在回答第8題：數學與邏輯
本題平均顯存：10437.94 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：10434.67 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：10436.14 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：10435.14 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：10434.07 MiB
正在回答第13題：翻譯
本題平均顯存：10431.99 MiB
正在回答第14題：翻譯
本題平均顯存：10432.12 MiB
正在回答第15題：翻譯
本題平均顯存：10432.16 MiB
正在回答第16題：翻譯
本題平均顯存：10432.57 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：10432.62 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：10432.95 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：10432.96 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：10432.58 MiB
整體平均顯存：10437.16 MiB
最高顯存：10538.36 MiB


Prompting Techniques(few shot + prompt用markdown形式)

In [10]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer, use_prompt_engineering=True)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_fewshot_markdown.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "FP16",
    "prompt_setting": "fewshot_markdown",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_fewshot_markdown_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：6601.38 MiB
正在回答第2題：知識理解
本題平均顯存：6609.35 MiB
正在回答第3題：知識理解
本題平均顯存：6607.03 MiB
正在回答第4題：知識理解
本題平均顯存：6601.74 MiB
正在回答第5題：數學與邏輯
本題平均顯存：6601.06 MiB
正在回答第6題：數學與邏輯
本題平均顯存：6601.91 MiB
正在回答第7題：數學與邏輯
本題平均顯存：6602.14 MiB
正在回答第8題：數學與邏輯
本題平均顯存：6605.80 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：6605.76 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：6605.82 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：6606.50 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：6605.36 MiB
正在回答第13題：翻譯
本題平均顯存：6601.15 MiB
正在回答第14題：翻譯
本題平均顯存：6601.21 MiB
正在回答第15題：翻譯
本題平均顯存：6601.43 MiB
正在回答第16題：翻譯
本題平均顯存：6602.02 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：6602.22 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：6602.46 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：6603.75 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：6602.85 MiB
整體平均顯存：6606.56 MiB
最高顯存：6918.23 MiB


使用NF4的4-bit權重量化

In [2]:
model_id = "Qwen/Qwen2.5-3B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config,
                                             torch_dtype=torch.float16, device_map="cuda:0")
model.eval()

print("是否以4-bit載入：", getattr(model, "is_loaded_in_4bit", False))

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.51s/it]


是否以4-bit載入： True


In [7]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_4bit.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "4-bit NF4 weights; FP16 compute",
    "prompt_setting": "baseline",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_4bit_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：2113.75 MiB
正在回答第2題：知識理解
本題平均顯存：2120.78 MiB
正在回答第3題：知識理解
本題平均顯存：2119.73 MiB
正在回答第4題：知識理解
本題平均顯存：2115.25 MiB
正在回答第5題：數學與邏輯
本題平均顯存：2115.08 MiB
正在回答第6題：數學與邏輯
本題平均顯存：2116.18 MiB
正在回答第7題：數學與邏輯
本題平均顯存：2116.01 MiB
正在回答第8題：數學與邏輯
本題平均顯存：2118.91 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：2118.19 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：2117.64 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：2118.62 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：2117.82 MiB
正在回答第13題：翻譯
本題平均顯存：2113.99 MiB
正在回答第14題：翻譯
本題平均顯存：2113.80 MiB
正在回答第15題：翻譯
本題平均顯存：2113.67 MiB
正在回答第16題：翻譯
本題平均顯存：2114.53 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：2114.73 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：2120.63 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：2115.65 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：2117.39 MiB
整體平均顯存：2118.31 MiB
最高顯存：2172.53 MiB


Prompting Techniques(few shot + prompt用markdown形式) + 使用NF4的4-bit權重量化

In [9]:
results = []
all_memory_samples = []
peak_memory_bytes = 0

(base_dir / "result").mkdir(exist_ok=True)

for item in questions:
    print(f"正在回答第{item['id']}題：{item['category']}")

    response, samples, peak = quiz1.answer_with_memory(question=item["question"], model=model, tokenizer=tokenizer)
    all_memory_samples.extend(samples)
    peak_memory_bytes = max(peak_memory_bytes, peak)

    average_mib = sum(samples) / len(samples) / (1024 ** 2)

    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "model_answer": response,
        "average_allocated_mib": average_mib,
        "peak_allocated_mib": peak / (1024 ** 2),
    })
    with open(base_dir / "result/quiz1/quiz1_4bit_fewshot_markdown.json", "w", encoding="utf-8",) as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"本題平均顯存：{average_mib:.2f} MiB")
    
summary = {
    "model": model_id,
    "precision": "4-bit NF4 weights; FP16 compute",
    "prompt_setting": "markdown_fewshot",
    "measurement": "PyTorch allocated memory",
    "sampling_interval_seconds": 0.05,
    "sample_count": len(all_memory_samples),
    "average_allocated_mib": (sum(all_memory_samples) / len(all_memory_samples) / (1024 ** 2)),
    "peak_allocated_mib": peak_memory_bytes / (1024 ** 2),
}

with open(base_dir / "result/quiz1/quiz1_4bit_fewshot_markdown_memory.json", "w", encoding="utf-8") as f: json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"整體平均顯存：{summary['average_allocated_mib']:.2f} MiB")
print(f"最高顯存：{summary['peak_allocated_mib']:.2f} MiB")    

正在回答第1題：知識理解
本題平均顯存：2113.75 MiB
正在回答第2題：知識理解
本題平均顯存：2120.56 MiB
正在回答第3題：知識理解
本題平均顯存：2119.62 MiB
正在回答第4題：知識理解
本題平均顯存：2114.80 MiB
正在回答第5題：數學與邏輯
本題平均顯存：2115.06 MiB
正在回答第6題：數學與邏輯
本題平均顯存：2116.59 MiB
正在回答第7題：數學與邏輯
本題平均顯存：2115.88 MiB
正在回答第8題：數學與邏輯
本題平均顯存：2118.68 MiB
正在回答第9題：閱讀理解與摘要
本題平均顯存：2118.35 MiB
正在回答第10題：閱讀理解與摘要
本題平均顯存：2117.81 MiB
正在回答第11題：閱讀理解與摘要
本題平均顯存：2118.54 MiB
正在回答第12題：閱讀理解與摘要
本題平均顯存：2117.79 MiB
正在回答第13題：翻譯
本題平均顯存：2113.80 MiB
正在回答第14題：翻譯
本題平均顯存：2113.85 MiB
正在回答第15題：翻譯
本題平均顯存：2115.95 MiB
正在回答第16題：翻譯
本題平均顯存：2114.38 MiB
正在回答第17題：指令與格式遵循
本題平均顯存：2114.70 MiB
正在回答第18題：指令與格式遵循
本題平均顯存：2120.36 MiB
正在回答第19題：指令與格式遵循
本題平均顯存：2115.90 MiB
正在回答第20題：指令與格式遵循
本題平均顯存：2114.80 MiB
整體平均顯存：2118.30 MiB
最高顯存：2172.53 MiB


# quiz 2 基礎